# CDG IA Sync Test — Qwen3 Forced Aligner

Prueba gratuita en Google Colab para obtener **timestamps palabra por palabra** desde una pista de voces + letra correcta.

**Usar GPU T4.**


In [ ]:
!pip -q install qwen-asr==0.0.6 mutagen==1.47.0
!apt-get -qq update && apt-get -qq install -y ffmpeg sox libsndfile1 > /dev/null
print('Dependencias listas')


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'SIN GPU')
if not torch.cuda.is_available():
    raise RuntimeError('Activa GPU T4 en Entorno de ejecución → Cambiar tipo de entorno de ejecución')


In [ ]:
from google.colab import files
uploaded = files.upload()
audio_path = next(iter(uploaded.keys()))
print('Audio:', audio_path)


In [ ]:
from mutagen import File as MutagenFile

audio = MutagenFile(audio_path, easy=False)
lyrics = ''
source = None
if audio is not None and getattr(audio, 'tags', None):
    for key in audio.tags.keys():
        lk = str(key).lower()
        if 'lyric' in lk or lk in {'uslt', '©lyr'}:
            value = audio.tags.get(key)
            text_value = getattr(value, 'text', value)
            if isinstance(text_value, (list, tuple)):
                text_value = '\n'.join(map(str, text_value))
            lyrics = str(text_value).strip()
            source = str(key)
            if lyrics:
                break

print('Letra embebida:', bool(lyrics), 'Fuente:', source)
print(lyrics[:1500] if lyrics else 'NO SE ENCONTRÓ LETRA')


Si no encontró letra, pega la letra correcta en la siguiente celda. Si sí la encontró, déjala vacía.


In [ ]:
manual_lyrics = ''''''
if manual_lyrics.strip():
    lyrics = manual_lyrics.strip()
if not lyrics.strip():
    raise RuntimeError('Falta la letra correcta para alinear')
print('Caracteres de letra:', len(lyrics))


In [ ]:
import torch
from qwen_asr import Qwen3ForcedAligner

aligner = Qwen3ForcedAligner.from_pretrained(
    'Qwen/Qwen3-ForcedAligner-0.6B',
    dtype=torch.float16,
    device_map='cuda:0',
)
print('Forced Aligner cargado')


In [ ]:
results = aligner.align(
    audio=audio_path,
    text=lyrics,
    language='Spanish',
)

items = results[0]
rows = []
for i, item in enumerate(items, 1):
    rows.append({
        'n': i,
        'word': item.text,
        'start': float(item.start_time),
        'end': float(item.end_time),
    })

print('Elementos temporizados:', len(rows))
rows[:30]


In [ ]:
import json
from pathlib import Path

out = Path('qwen_word_timestamps.json')
out.write_text(json.dumps(rows, ensure_ascii=False, indent=2), encoding='utf-8')
print(out.resolve())
files.download(str(out))
